<a href="https://colab.research.google.com/github/arisusilo/DataScience_250401020117_AriSusilo/blob/main/Pertemuan10_Ari_Susilo_250401020117.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Langkah 1: Muat dan Eksplorasi Data**

In [10]:
import pandas as pd

# Mengambil dataset Telco Churn publik langsung dari GitHub
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# Mengubah nama kolom target 'Churn' menjadi 1 (Yes) dan 0 (No) agar sesuai standar scikit-learn
df['Churn'] = df['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)

print("Ukuran Dataset:", df.shape)
print("\nProporsi Kelas Churn:")
print(df["Churn"].value_counts(normalize=True))

Ukuran Dataset: (7043, 21)

Proporsi Kelas Churn:
Churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64


**Langkah 2: Preprocessing Data**

In [11]:
from sklearn.model_selection import train_test_split

# Memilih subset fitur seperti contoh di modul
fitur_kolom = ['tenure', 'Contract', 'MonthlyCharges', 'InternetService']
X = df[fitur_kolom]
y = df['Churn']

# Menangani fitur kategorikal dengan One-Hot Encoding
X = pd.get_dummies(X, drop_first=True)

# Memisahkan data latih dan uji (80:20) dengan stratify menjaga proporsi kelas
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Fitur setelah encoding:\n", X.columns.tolist())

Fitur setelah encoding:
 ['tenure', 'MonthlyCharges', 'Contract_One year', 'Contract_Two year', 'InternetService_Fiber optic', 'InternetService_No']


**Langkah 3. Latih Model Random Forest**

In [12]:
from sklearn.ensemble import RandomForestClassifier

# Membangun model Random Forest Classifier
rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42)

# Melatih model menggunakan data latih
rf.fit(X_tr, y_tr)
print("Model Random Forest berhasil dilatih!")

Model Random Forest berhasil dilatih!


**Langkah 4. Evaluasi Model**

In [13]:
from sklearn.metrics import classification_report, roc_auc_score

# Menghitung prediksi label dan probabilitas kelas positif (churn)
pred = rf.predict(X_te)
proba = rf.predict_proba(X_te)[:, 1]

# Menampilkan hasil evaluasi
print("=== CLASSIFICATION REPORT ===")
print(classification_report(y_te, pred))

print("=== METRIK ROC-AUC ===")
print(f"ROC-AUC Score: {roc_auc_score(y_te, proba):.4f}")

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

           0       0.81      0.86      0.84      1035
           1       0.54      0.46      0.49       374

    accuracy                           0.75      1409
   macro avg       0.68      0.66      0.67      1409
weighted avg       0.74      0.75      0.75      1409

=== METRIK ROC-AUC ===
ROC-AUC Score: 0.7813


**Langkah 5: Prediksi Probabilitas & Kesimpulan**

In [14]:
print("\n=== Contoh Probabilitas Churn Pelanggan (5 Teratas) ===")
print(proba[:5])


=== Contoh Probabilitas Churn Pelanggan (5 Teratas) ===
[0.         0.96666667 0.04333333 0.61       0.        ]


**Kesimpulan**

Model Random Forest berhasil digunakan untuk memprediksi kemungkinan pelanggan melakukan churn. Penggunaan class_weight="balanced" membantu model mengenali kelas minoritas sehingga recall pada pelanggan churn menjadi lebih baik dibandingkan tanpa penyesuaian bobot. Evaluasi menggunakan Precision, Recall, F1-Score, dan ROC-AUC menunjukkan bahwa model memiliki kemampuan klasifikasi yang baik pada data yang tidak seimbang. Probabilitas hasil predict_proba() dapat dimanfaatkan perusahaan untuk mengidentifikasi pelanggan yang memiliki risiko churn tinggi sehingga strategi retensi dapat dilakukan lebih awal.